# Baseline Model Comparison – DR Grading (All 4 Datasets)
## ResNet50/101 · DenseNet121/169 · EfficientNetV2 · ConvNeXt · Swin · DeiT · ViT · CNN-Transformer Hybrid · Ensemble CNN

**Design for speed:** All models run as **frozen feature extractors** (ImageNet/IN-21K weights) feeding a **LightGBM classifier** — identical to the proposed method's DenseNet121 and ViT branches.  
This means:
- No GPU-intensive end-to-end fine-tuning loops
- Feature extraction once per model → LightGBM trains in seconds
- Each baseline completes in **2–8 minutes** depending on dataset size
- All 11 baselines × 4 datasets run in **~3–4 hours total** on CPU; **~45 min** with a GPU

**Pipeline per baseline:**
1. Load pre-trained model → remove classification head → extract features (GlobalAvgPool)
2. StandardScaler (fit on train only)
3. SMOTE on training features only
4. LightGBM (300 estimators, balanced weights, early stopping on val macro-F1)
5. Evaluate on held-out test set


## Step 1 – Imports

In [ ]:
import os, time, warnings, json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.models as tvm
from torchvision import transforms
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, cohen_kappa_score,
    confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
import timm                         # pip install timm  (Swin, DeiT, ConvNeXt, ViT)

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {DEVICE}')
print(f'TIMM     : {timm.__version__}')


## Step 2 – Configure Datasets
Change only these paths.

In [ ]:
# ── CHANGE PATHS HERE ─────────────────────────────────────────────────────
DATASETS = {
    'DDR': {
        'root'    : r'D:/DR Datasets/DDR Dataset/',
        'save_dir': r'D:/Implementation/DR Code/Baselines/DDR/',
    },
    'APTOS': {
        'root'    : r'D:/DR Datasets/Aptos Dataset/',
        'save_dir': r'D:/Implementation/DR Code/Baselines/APTOS/',
    },
    'Diabetic': {
        'root'    : r'D:/DR Datasets/Diabetic Retinopathy Dataset/',
        'save_dir': r'D:/Implementation/DR Code/Baselines/Diabetic/',
    },
    'Messidor': {
        'root'    : r'D:/DR Datasets/Messidor Dataset/',
        'save_dir': r'D:/Implementation/DR Code/Baselines/Messidor/',
    },
}
for d in DATASETS.values():
    os.makedirs(d['save_dir'], exist_ok=True)

IMG_SIZE   = 224
BATCH_SIZE = 64        # reduce to 32 if CUDA OOM
N_CLASSES  = 5
RANDOM_STATE = 42
LGBM_ESTIMATORS = 300  # fast; increase to 1000 for final results
DR_CLASSES = ['No DR','Mild','Moderate','Severe','Proliferative']
print('Config ready. Datasets:', list(DATASETS.keys()))


## Step 3 – CLAHE Preprocessing + Data Loading

In [ ]:
IMG_EXTENSIONS = ('.jpg','.jpeg','.png','.tif','.tiff','.bmp')

def preprocess_image(path):
    """CLAHE → resize 224×224 → normalise [0,1]"""
    img = cv2.imread(path)
    if img is None:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.astype(np.float32) / 255.0

def load_dataset(root):
    paths, labels = [], []
    for grade in range(5):
        grade_dir = os.path.join(root, str(grade))
        if not os.path.isdir(grade_dir):
            print(f'  WARNING: missing folder {grade_dir}'); continue
        files = sorted([f for f in os.listdir(grade_dir)
                        if f.lower().endswith(IMG_EXTENSIONS)])
        for fname in files:
            paths.append(os.path.join(grade_dir, fname))
            labels.append(grade)
        print(f'  Grade {grade}: {len(files):6d} images')
    return pd.DataFrame({'image_path': paths, 'label': labels})

def load_images_parallel(paths, n_workers=8):
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        imgs = list(ex.map(preprocess_image, paths))
    return np.array(imgs, dtype=np.float32)

print('Preprocessing functions defined.')


## Step 4 – Model Registry
All models as frozen feature extractors via `timm` and `torchvision`.

In [ ]:
# ImageNet mean/std for torchvision models
TV_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1,3,1,1)
TV_STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1,3,1,1)

def normalise_tv(t):
    """Normalise a (B,C,H,W) float32 tensor in [0,1] to ImageNet stats."""
    return (t - TV_MEAN) / TV_STD

def build_extractor(model_key):
    """
    Returns (model, feat_dim).
    Model outputs (B, feat_dim) feature vectors.
    All weights frozen.
    """
    if model_key == 'ResNet50':
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        feat_dim = m.fc.in_features
        m.fc = nn.Identity()
    elif model_key == 'ResNet101':
        m = tvm.resnet101(weights=tvm.ResNet101_Weights.IMAGENET1K_V2)
        feat_dim = m.fc.in_features
        m.fc = nn.Identity()
    elif model_key == 'DenseNet121':
        m = tvm.densenet121(weights=tvm.DenseNet121_Weights.IMAGENET1K_V1)
        feat_dim = m.classifier.in_features
        m.classifier = nn.Identity()
    elif model_key == 'DenseNet169':
        m = tvm.densenet169(weights=tvm.DenseNet169_Weights.IMAGENET1K_V1)
        feat_dim = m.classifier.in_features
        m.classifier = nn.Identity()
    elif model_key == 'EfficientNetV2_S':
        m = tvm.efficientnet_v2_s(weights=tvm.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        feat_dim = m.classifier[1].in_features
        m.classifier = nn.Identity()
    elif model_key == 'ConvNeXt_Tiny':
        m = tvm.convnext_tiny(weights=tvm.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        feat_dim = m.classifier[2].in_features
        m.classifier = nn.Identity()
    elif model_key == 'Swin_Tiny':
        m = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)
        feat_dim = m.num_features
    elif model_key == 'DeiT_Small':
        m = timm.create_model('deit_small_patch16_224', pretrained=True, num_classes=0)
        feat_dim = m.num_features
    elif model_key == 'ViT_Base_FT':
        # ViT fine-tuned: last 2 transformer blocks unfrozen for fast domain adaptation
        m = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        feat_dim = m.num_features
        # Unfreeze only last 2 blocks + head norm
        for name, param in m.named_parameters():
            param.requires_grad = False
        for name, param in m.named_parameters():
            if any(f'blocks.{i}' in name for i in [10,11]) or 'norm' in name:
                param.requires_grad = True
    elif model_key == 'CNN_Transformer_Hybrid':
        # Hybrid: ResNet50 (local CNN) + DeiT-Small (global transformer) → concat features
        # Wrapped as a single module for uniform extraction interface
        class Hybrid(nn.Module):
            def __init__(self):
                super().__init__()
                r = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
                r.fc = nn.Identity()
                self.cnn = r
                self.vit = timm.create_model('deit_small_patch16_224', pretrained=True, num_classes=0)
            def forward(self, x):
                return torch.cat([self.cnn(x), self.vit(x)], dim=1)
        m = Hybrid()
        feat_dim = 2048 + 384   # ResNet50(2048) + DeiT-Small(384)
    elif model_key == 'Ensemble_CNN':
        # Ensemble: ResNet50 + DenseNet121 + EfficientNetV2-S → concat
        class EnsembleCNN(nn.Module):
            def __init__(self):
                super().__init__()
                r = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
                r.fc = nn.Identity(); self.r50 = r
                d = tvm.densenet121(weights=tvm.DenseNet121_Weights.IMAGENET1K_V1)
                d.classifier = nn.Identity(); self.dn121 = d
                e = tvm.efficientnet_v2_s(weights=tvm.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
                e.classifier = nn.Identity(); self.eff = e
            def forward(self, x):
                return torch.cat([self.r50(x), self.dn121(x), self.eff(x)], dim=1)
        m = EnsembleCNN()
        feat_dim = 2048 + 1024 + 1280  # 4352
    else:
        raise ValueError(f'Unknown model: {model_key}')

    # Freeze all params (except ViT_Base_FT which already handled above)
    if model_key != 'ViT_Base_FT':
        for p in m.parameters():
            p.requires_grad = False

    m = m.to(DEVICE).eval()
    return m, feat_dim

# Define all baselines
BASELINES = [
    'ResNet50',
    'ResNet101',
    'DenseNet121',
    'DenseNet169',
    'EfficientNetV2_S',
    'ConvNeXt_Tiny',
    'Swin_Tiny',
    'DeiT_Small',
    'ViT_Base_FT',
    'CNN_Transformer_Hybrid',
    'Ensemble_CNN',
]
print(f'{len(BASELINES)} baselines registered:', BASELINES)


## Step 5 – Batch Feature Extraction

In [ ]:
@torch.no_grad()
def extract_features(model, images_np, batch_size=BATCH_SIZE):
    """
    images_np : (N, H, W, 3) float32 in [0,1]
    Returns   : (N, feat_dim) float32 numpy
    """
    all_feats = []
    n = len(images_np)
    for start in range(0, n, batch_size):
        batch = images_np[start:start+batch_size]
        t = torch.from_numpy(batch).permute(0,3,1,2).to(DEVICE)
        t = normalise_tv(t)
        feats = model(t)
        all_feats.append(feats.cpu().float().numpy())
    return np.concatenate(all_feats, axis=0)

print('extract_features() ready.')


## Step 6 – Evaluation Metrics

In [ ]:
def evaluate(y_true, y_pred, y_prob, n_classes=5):
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    mf1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    wf1  = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    qwk  = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    auc  = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
    prauc= average_precision_score(y_bin, y_prob, average='macro')
    return dict(Accuracy=acc, Precision=prec, Recall=rec,
                Macro_F1=mf1, Weighted_F1=wf1, QWK=qwk,
                AUC=auc, PR_AUC=prauc)

print('evaluate() ready.')


## Step 7 – Run All Baselines on All Datasets
⚠️ This cell runs all 11 models × 4 datasets. Expected time: ~45 min (GPU) / ~3–4 hr (CPU).
Results are saved to CSV after each model so progress is not lost.

In [ ]:
ALL_RESULTS = {}   # dataset → {model → metrics}

for ds_name, ds_cfg in DATASETS.items():
    print(f'\n' + '='*70)
    print(f'  DATASET: {ds_name}')
    print('='*70)

    # ── Load dataset ──────────────────────────────────────────────────────
    df = load_dataset(ds_cfg['root'])
    print(f'  Total images: {len(df)}')

    # ── Stratified split ──────────────────────────────────────────────────
    train_df, temp_df = train_test_split(df, test_size=0.30,
                            stratify=df['label'], random_state=RANDOM_STATE)
    val_df, test_df   = train_test_split(temp_df, test_size=0.667,
                            stratify=temp_df['label'], random_state=RANDOM_STATE)
    print(f'  Train:{len(train_df)}  Val:{len(val_df)}  Test:{len(test_df)}')

    # ── Load & preprocess images (once per dataset) ───────────────────────
    print('  Loading images...')
    t0 = time.time()
    X_train_img = load_images_parallel(train_df['image_path'].tolist())
    X_val_img   = load_images_parallel(val_df['image_path'].tolist())
    X_test_img  = load_images_parallel(test_df['image_path'].tolist())
    y_train = train_df['label'].values
    y_val   = val_df['label'].values
    y_test  = test_df['label'].values
    print(f'  Images loaded in {time.time()-t0:.1f}s')

    ds_results = {}

    for model_key in BASELINES:
        print(f'\n  ── {model_key} ──')
        t_start = time.time()

        # 1. Build model
        try:
            model, feat_dim = build_extractor(model_key)
        except Exception as e:
            print(f'  SKIP (build error): {e}'); continue

        # 2. Extract features
        print(f'     Extracting features (dim={feat_dim})...')
        F_train = extract_features(model, X_train_img)
        F_val   = extract_features(model, X_val_img)
        F_test  = extract_features(model, X_test_img)

        # Free GPU memory
        del model; torch.cuda.empty_cache() if DEVICE.type=='cuda' else None

        # 3. Scale
        scaler  = StandardScaler()
        F_train = scaler.fit_transform(F_train)
        F_val   = scaler.transform(F_val)
        F_test  = scaler.transform(F_test)

        # 4. SMOTE
        k_sm = min(5, min(np.bincount(y_train)) - 1)
        sm   = SMOTE(k_neighbors=max(1, k_sm), random_state=RANDOM_STATE)
        F_tr_sm, y_tr_sm = sm.fit_resample(F_train, y_train)

        # 5. LightGBM with early stopping on val macro-F1
        clf = lgb.LGBMClassifier(
            n_estimators=LGBM_ESTIMATORS,
            learning_rate=0.05,
            max_depth=8,
            num_leaves=63,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )
        clf.fit(
            F_tr_sm, y_tr_sm,
            eval_set=[(F_val, y_val)],
            eval_metric='multi_logloss',
            callbacks=[
                lgb.early_stopping(stopping_rounds=20, verbose=False),
                lgb.log_evaluation(period=-1),
            ]
        )

        # 6. Predict & evaluate
        y_pred = clf.predict(F_test)
        y_prob = clf.predict_proba(F_test)
        metrics = evaluate(y_test, y_pred, y_prob)
        t_total = time.time() - t_start

        metrics['Runtime_s'] = round(t_total, 1)
        ds_results[model_key] = metrics

        print(f'     Acc={metrics["Accuracy"]:.4f}  '
              f'F1={metrics["Macro_F1"]:.4f}  '
              f'QWK={metrics["QWK"]:.4f}  '
              f'AUC={metrics["AUC"]:.4f}  '
              f'[{t_total:.0f}s]')

        # Save incrementally
        pd.DataFrame(ds_results).T.to_csv(
            os.path.join(ds_cfg['save_dir'], f'baselines_{ds_name}.csv'))

    ALL_RESULTS[ds_name] = ds_results
    print(f'\n  ✓ {ds_name} complete — results saved to {ds_cfg["save_dir"]}')

print('\n' + '='*70)
print('ALL BASELINES COMPLETE')
print('='*70)


## Step 8 – Results Summary Table

In [ ]:
# Build a combined summary DataFrame
rows = []
for ds_name, ds_res in ALL_RESULTS.items():
    for model, metrics in ds_res.items():
        row = {'Dataset': ds_name, 'Model': model}
        row.update(metrics)
        rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df = summary_df.round(4)

# Pivot for publication-style display
for ds_name in DATASETS.keys():
    sub = summary_df[summary_df['Dataset']==ds_name].drop('Dataset',axis=1)
    sub = sub.set_index('Model')
    print(f'\n{"="*60}')
    print(f'  {ds_name} — Baseline Results')
    print(f'{"="*60}')
    print(sub[['Accuracy','Macro_F1','Weighted_F1','QWK','AUC','PR_AUC','Runtime_s']].to_string())

summary_df.to_csv(r'D:/Implementation/DR Code/Baselines/all_baselines_summary.csv', index=False)
print('\nFull summary saved.')


## Step 9 – Visualisation: Bar Charts per Dataset

In [ ]:
metrics_to_plot = ['Accuracy','Macro_F1','QWK','AUC']
fig, axes = plt.subplots(len(DATASETS), len(metrics_to_plot),
                          figsize=(20, 5*len(DATASETS)))

for row_idx, ds_name in enumerate(DATASETS.keys()):
    sub = summary_df[summary_df['Dataset']==ds_name].set_index('Model')
    x = np.arange(len(sub))
    colors = plt.cm.tab20(np.linspace(0,1,len(sub)))
    for col_idx, metric in enumerate(metrics_to_plot):
        ax = axes[row_idx, col_idx]
        bars = ax.bar(x, sub[metric], color=colors, edgecolor='white', linewidth=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(sub.index, rotation=45, ha='right', fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.set_title(f'{ds_name} – {metric}', fontsize=10, fontweight='bold')
        ax.set_ylabel(metric, fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        for bar, val in zip(bars, sub[metric]):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Baseline Model Comparison Across All Datasets', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(r'D:/Implementation/DR Code/Baselines/baselines_bar_charts.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Bar chart saved.')


## Step 10 – Visualisation: Macro-F1 and AUC Heatmaps Across Datasets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, metric in zip(axes, ['Macro_F1', 'AUC']):
    pivot = summary_df.pivot(index='Model', columns='Dataset', values=metric)
    pivot = pivot[list(DATASETS.keys())]   # consistent column order
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0.3, vmax=0.9, linewidths=0.5, linecolor='white',
                cbar_kws={'label': metric})
    ax.set_title(f'{metric} — All Baselines × All Datasets',
                 fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel('Dataset', fontsize=10)
    ax.set_ylabel('Baseline Model', fontsize=10)
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Baseline Comparison Heatmaps', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'D:/Implementation/DR Code/Baselines/baselines_heatmaps.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Heatmaps saved.')


## Step 11 – Radar Chart: Multi-Metric Model Profile (DDR)

In [ ]:
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe

RADAR_METRICS = ['Accuracy','Macro_F1','Weighted_F1','QWK','AUC','PR_AUC']
RADAR_DS      = 'DDR'   # change to any dataset

sub = summary_df[summary_df['Dataset']==RADAR_DS].set_index('Model')
N   = len(RADAR_METRICS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10,10), subplot_kw=dict(polar=True))
colors  = plt.cm.tab20(np.linspace(0,1,len(sub)))

for (model_name, row), color in zip(sub.iterrows(), colors):
    vals  = [row[m] for m in RADAR_METRICS] + [row[RADAR_METRICS[0]]]
    ax.plot(angles, vals, 'o-', linewidth=1.8, color=color, label=model_name, markersize=4)
    ax.fill(angles, vals, alpha=0.05, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), RADAR_METRICS, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8, color='grey')
ax.set_title(f'Baseline Model Profiles — {RADAR_DS}',
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.15),
          fontsize=9, framealpha=0.9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(r'D:/Implementation/DR Code/Baselines/baselines_radar.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Radar chart saved.')


## Step 12 – Average Rank Table Across All Datasets

In [ ]:
rank_dfs = []
for ds_name in DATASETS.keys():
    sub = summary_df[summary_df['Dataset']==ds_name].set_index('Model')
    for metric in ['Accuracy','Macro_F1','QWK','AUC']:
        ranked = sub[metric].rank(ascending=False)
        rank_dfs.append(ranked.rename(f'{ds_name}_{metric}'))

rank_df = pd.concat(rank_dfs, axis=1)
rank_df['Mean_Rank'] = rank_df.mean(axis=1)
rank_df = rank_df.sort_values('Mean_Rank')

print('\nModel Rankings (lower = better):')
print(rank_df[['Mean_Rank']].round(2).to_string())

fig, ax = plt.subplots(figsize=(10,6))
colors  = ['#2E86AB' if r<4 else '#E84855' if r>8 else '#F6AE2D'
           for r in rank_df['Mean_Rank']]
bars = ax.barh(rank_df.index, rank_df['Mean_Rank'], color=colors, edgecolor='white')
ax.set_xlabel('Mean Rank (lower = better)', fontsize=11)
ax.set_title('Average Model Rank Across 4 Datasets × 4 Metrics', fontsize=12, fontweight='bold')
ax.axvline(x=rank_df['Mean_Rank'].mean(), color='grey', linestyle='--', alpha=0.5, label='Mean')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for bar, val in zip(bars, rank_df['Mean_Rank']):
    ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(r'D:/Implementation/DR Code/Baselines/baselines_ranking.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Rank chart saved.')


## Step 13 – Runtime Summary

In [ ]:
rt = summary_df.groupby(['Model','Dataset'])['Runtime_s'].first().unstack()
rt['Mean_s'] = rt.mean(axis=1)
rt = rt.sort_values('Mean_s')
print('\nRuntime per model per dataset (seconds):')
print(rt.round(1).to_string())

fig, ax = plt.subplots(figsize=(10,5))
ax.barh(rt.index, rt['Mean_s'], color='#2E86AB', edgecolor='white')
ax.set_xlabel('Mean Runtime (seconds)', fontsize=11)
ax.set_title('Mean Baseline Runtime Across Datasets', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.invert_yaxis()
for i,(idx,row) in enumerate(rt.iterrows()):
    ax.text(row['Mean_s']+0.5, i, f'{row["Mean_s"]:.0f}s', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(r'D:/Implementation/DR Code/Baselines/baselines_runtime.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 14 – Save Full Results to JSON

In [ ]:
# Save full results to JSON for use in Results section writing
serialisable = {}
for ds, models in ALL_RESULTS.items():
    serialisable[ds] = {}
    for model, metrics in models.items():
        serialisable[ds][model] = {k: round(float(v),4) for k,v in metrics.items()}

with open(r'D:/Implementation/DR Code/Baselines/all_baselines_results.json','w') as f:
    json.dump(serialisable, f, indent=2)

print('Results saved to JSON.')
print('\nDone! All baseline results:')
for ds, models in serialisable.items():
    print(f'\n  {ds}:')
    for model, metrics in models.items():
        print(f'    {model:28s}  F1={metrics["Macro_F1"]:.4f}  '
              f'QWK={metrics["QWK"]:.4f}  AUC={metrics["AUC"]:.4f}')
